In [6]:
import io
import os
import re
import gzip
import requests
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


# ============================================================
# USER SETTINGS
# ============================================================

START = "2024-09-24"
END   = "2024-09-28"
YEAR  = 2024

OUT_NC     = "ndbc_bulk_with_metadata.nc"
PLOTDIR    = "F:/crs/proj/2025_NOPP_comparison/helene_waves/comparison_plots/"

USE_PERIOD = "dpd"          # "dpd" or "apd"
TIME_TOL   = "30min"        # nearest-time tolerance for comparisons
MAKE_PLOTS = True


# ============================================================
# BASIC UTILITIES
# ============================================================

NA_VALUES = ["MM", "999", "999.0", "9999", "99.0", "99.00", "99999", "999999"]


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def wrap360(deg):
    return np.mod(np.asarray(deg), 360.0)


def circ_diff_deg(a, b):
    """
    Signed shortest angular difference a-b in degrees.
    """
    return (np.asarray(a) - np.asarray(b) + 180.0) % 360.0 - 180.0


def rmse(a):
    a = np.asarray(a, dtype=float)
    return np.sqrt(np.nanmean(a**2))


def decode_string_array(arr):
    """
    Decode:
      - fixed-width byte strings
      - object strings
      - 2D char arrays
    to 1D numpy array of Python strings.
    """
    arr = np.asarray(arr)

    if arr.ndim == 2:
        out = []
        for row in arr:
            chars = []
            for x in row:
                if isinstance(x, (bytes, np.bytes_)):
                    chars.append(x.decode("utf-8", errors="ignore"))
                else:
                    chars.append(str(x))
            out.append("".join(chars).strip().strip("\x00"))
        return np.array(out, dtype=object)

    out = []
    for x in arr:
        if isinstance(x, (bytes, np.bytes_)):
            out.append(x.decode("utf-8", errors="ignore").strip().strip("\x00"))
        else:
            out.append(str(x).strip().strip("\x00"))
    return np.array(out, dtype=object)


def extract_ndbc_id(text):
    """
    Extract a 5-digit NDBC ID like 42002 from text.
    """
    s = str(text).strip()
    m = re.search(r"\b(4\d{4})\b", s)
    return m.group(1) if m else None


def get_time_name(ds):
    for v in ["time", "ocean_time"]:
        if v in ds.coords or v in ds.variables:
            return v
    raise KeyError(f"Could not find time coordinate in {list(ds.variables)}")


# ============================================================
# READ LOCAL BUOY METADATA CSV
# ============================================================

def read_buoy_metadata(meta_csv):
    """
    Read buoy metadata from local CSV.

    Expected columns:
      name,longname,longitude,latitude,depth (m)
    """
    meta = pd.read_csv(meta_csv)

    # Clean column names a bit
    meta.columns = [c.strip() for c in meta.columns]

    required = ["name", "longname", "longitude", "latitude", "depth (m)"]
    missing = [c for c in required if c not in meta.columns]
    if missing:
        raise ValueError(f"Missing required metadata columns: {missing}")

    meta = meta.copy()
    meta["station_id"] = meta["name"].astype(str).str.strip()
    meta["station_name"] = meta["longname"].astype(str).str.strip()
    meta["longitude"] = pd.to_numeric(meta["longitude"], errors="coerce")
    meta["latitude"] = pd.to_numeric(meta["latitude"], errors="coerce")
    meta["water_depth_m"] = pd.to_numeric(meta["depth (m)"], errors="coerce")

    # Keep only useful columns
    meta = meta[[
        "station_id",
        "station_name",
        "longitude",
        "latitude",
        "water_depth_m",
    ]].drop_duplicates(subset="station_id")

    return meta


# ============================================================
# DOWNLOAD AND PARSE NDBC STDMET
# ============================================================

def _parse_ndbc_stdmet_text(text):
    """
    Parse one NDBC historical stdmet text file into a DataFrame.
    """
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    if not lines:
        raise ValueError("Empty file")

    header_line = lines[0].lstrip("#").strip()
    cols = header_line.split()

    data_start = 1
    if len(lines) > 1:
        second = lines[1].split()
        nonnumeric = 0
        for tok in second:
            try:
                float(tok)
            except Exception:
                nonnumeric += 1
        if len(second) == len(cols) and nonnumeric >= max(3, len(second) // 3):
            data_start = 2

    data_text = "\n".join(lines[data_start:])
    if not data_text.strip():
        return pd.DataFrame(columns=cols)

    df = pd.read_csv(
        io.StringIO(data_text),
        sep=r"\s+",
        names=cols,
        na_values=NA_VALUES,
        engine="python",
    )

    year_col = None
    for candidate in ["YYYY", "#YYYY", "YY", "#YY"]:
        if candidate in df.columns:
            year_col = candidate
            break
    if year_col is None:
        raise ValueError(f"Could not find year column in columns: {list(df.columns)}")

    for c in ["MM", "DD", "hh"]:
        if c not in df.columns:
            raise ValueError(f"Missing required time column: {c}")

    minute_col = "mm" if "mm" in df.columns else None

    year = df[year_col].astype(int).to_numpy()
    if "YY" in year_col:
        year = np.where(year < 100, 2000 + year, year)

    month = df["MM"].astype(int).to_numpy()
    day = df["DD"].astype(int).to_numpy()
    hour = df["hh"].astype(int).to_numpy()
    minute = df[minute_col].astype(int).to_numpy() if minute_col else np.zeros(len(df), dtype=int)

    dt = pd.to_datetime(
        {
            "year": year,
            "month": month,
            "day": day,
            "hour": hour,
            "minute": minute,
        },
        errors="coerce",
        utc=True,
    )

    df.insert(0, "time", dt)
    df = df.dropna(subset=["time"]).reset_index(drop=True)
    return df


def download_ndbc_stdmet_station_year(station_id, year=2024, timeout=60):
    """
    Download one NDBC historical stdmet file and parse it.
    """
    url = f"https://www.ndbc.noaa.gov/data/historical/stdmet/{station_id}h{year}.txt.gz"
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()

    with gzip.GzipFile(fileobj=io.BytesIO(r.content)) as gz:
        text = gz.read().decode("utf-8", errors="replace")

    df = _parse_ndbc_stdmet_text(text)
    df["station_id"] = station_id
    return df


def subset_date_range_inclusive(df, start, end, time_col="time"):
    """
    Inclusive UTC date filter.
    """
    start_ts = pd.Timestamp(start, tz="UTC")
    end_ts = pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1) - pd.Timedelta(microseconds=1)
    return df.loc[(df[time_col] >= start_ts) & (df[time_col] <= end_ts)].copy()


def download_ndbc_bulk_for_list(meta, start, end, year=2024):
    """
    Download NDBC stdmet data for every station listed in metadata.
    Returns:
      combined dataframe
      dict of per-station dataframes
    """
    station_ids = meta["station_id"].tolist()
    per_station = {}

    for sid in station_ids:
        try:
            df = download_ndbc_stdmet_station_year(sid, year=year)
            df = subset_date_range_inclusive(df, start=start, end=end)
            df = df.merge(meta, on="station_id", how="left")
            per_station[sid] = df
            print(f"{sid}: {len(df)} records")
        except Exception as e:
            print(f"{sid}: FAILED -> {e}")
            per_station[sid] = pd.DataFrame()

    combined = pd.concat(
        [df for df in per_station.values() if len(df) > 0],
        ignore_index=True,
        sort=False,
    ) if any(len(df) > 0 for df in per_station.values()) else pd.DataFrame()

    if len(combined) > 0:
        preferred = [
            "station_id", "station_name", "time",
            "latitude", "longitude", "water_depth_m",
            "WDIR", "WSPD", "GST",
            "WVHT", "DPD", "APD", "MWD",
            "PRES", "ATMP", "WTMP", "DEWP", "VIS", "PTDY", "TIDE",
        ]
        cols = [c for c in preferred if c in combined.columns] + [c for c in combined.columns if c not in preferred]
        combined = combined[cols].sort_values(["station_id", "time"]).reset_index(drop=True)

    return combined, per_station


# ============================================================
# SAVE PER-BUOY CSV FILES
# ============================================================

def save_station_csvs(per_station, outdir):
    ensure_dir(outdir)

    for sid, df in per_station.items():
        out_csv = os.path.join(outdir, f"ndbc_{sid}.csv")
        if len(df) == 0:
            pd.DataFrame().to_csv(out_csv, index=False)
        else:
            tmp = df.copy()
            if "time" in tmp.columns:
                tmp["time"] = pd.to_datetime(tmp["time"], utc=True).dt.strftime("%Y-%m-%dT%H:%M:%SZ")
            tmp.to_csv(out_csv, index=False)

    print(f"Saved per-station CSV files to: {outdir}")


# ============================================================
# BUILD AND SAVE NETCDF
# ============================================================

def build_ndbc_xarray(df, meta):
    """
    Build xarray Dataset from downloaded NDBC data plus metadata.
    Includes all stations in metadata, even if some have no data.
    """
    station_ids = meta["station_id"].tolist()

    if len(df) > 0:
        df = df.copy()
        df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert(None)
        time = np.sort(df["time"].dropna().unique()).astype("datetime64[ns]")
    else:
        time = np.array([], dtype="datetime64[ns]")

    ntime = len(time)
    nsta = len(station_ids)

    meta2 = meta.set_index("station_id").reindex(station_ids)

    ds = xr.Dataset(
        coords={
            "time": ("time", time),
            "station": np.arange(nsta, dtype=np.int32),
        }
    )

    ds["station_id"] = ("station", np.asarray(station_ids, dtype="S8"))
    ds["station_name"] = ("station", np.asarray(meta2["station_name"].fillna("").tolist(), dtype="S128"))
    ds["longitude"] = ("station", meta2["longitude"].to_numpy(dtype=np.float32))
    ds["latitude"] = ("station", meta2["latitude"].to_numpy(dtype=np.float32))
    ds["water_depth_m"] = ("station", meta2["water_depth_m"].to_numpy(dtype=np.float32))

    ds["longitude"].attrs["units"] = "degrees_east"
    ds["latitude"].attrs["units"] = "degrees_north"
    ds["water_depth_m"].attrs["units"] = "m"

    time_index = pd.Index(time)
    station_lookup = {sid: j for j, sid in enumerate(station_ids)}

    var_map = {
        "WVHT": ("wave_height", "m"),
        "DPD": ("dominant_period", "s"),
        "APD": ("average_period", "s"),
        "MWD": ("mean_wave_direction", "deg"),
        "WSPD": ("wind_speed", "m/s"),
        "WDIR": ("wind_direction", "deg"),
        "GST": ("wind_gust", "m/s"),
        "PRES": ("air_pressure", "hPa"),
        "ATMP": ("air_temperature", "degC"),
        "WTMP": ("water_temperature", "degC"),
    }

    for src_name, (out_name, units) in var_map.items():
        arr = np.full((ntime, nsta), np.nan, dtype=np.float32)

        if len(df) > 0 and src_name in df.columns:
            for sid, g in df.groupby("station_id"):
                if sid not in station_lookup:
                    continue
                j = station_lookup[sid]

                gg = (
                    g[["time", src_name]]
                    .dropna()
                    .drop_duplicates(subset="time")
                    .sort_values("time")
                )
                if len(gg) == 0:
                    continue

                it = time_index.get_indexer(gg["time"].to_numpy().astype("datetime64[ns]"))
                good = it >= 0
                arr[it[good], j] = gg[src_name].to_numpy(dtype=np.float32)[good]

        ds[out_name] = (("time", "station"), arr)
        ds[out_name].attrs["units"] = units
        ds[out_name].attrs["source_name"] = src_name

    return ds


def save_ndbc_to_netcdf(df, meta, out_nc):
    ds = build_ndbc_xarray(df, meta)
    try:
        ds.to_netcdf(out_nc, engine="netcdf4")
    except Exception:
        ds.to_netcdf(out_nc)
    print(f"Saved NetCDF: {out_nc}")
    return ds


# ============================================================
# READ NDBC NETCDF FOR COMPARISON
# ============================================================

def read_ndbc_bulk_nc(nc_path):
    ds0 = xr.open_dataset(nc_path, decode_cf=True)
    time_name = get_time_name(ds0)

    out = xr.Dataset(
        coords={
            "time": ds0[time_name].values,
            "station": np.arange(ds0.sizes["station"]),
        }
    )

    out["station_id"] = ("station", decode_string_array(ds0["station_id"].values))
    out["station_name"] = ("station", decode_string_array(ds0["station_name"].values))

    for v in ["longitude", "latitude", "water_depth_m"]:
        if v in ds0:
            out[v] = ("station", ds0[v].values)

    varmap = {
        "wave_height": "wvht_buoy",
        "dominant_period": "dpd_buoy",
        "average_period": "apd_buoy",
        "mean_wave_direction": "mwd_from_buoy",
    }

    for src, dst in varmap.items():
        if src in ds0:
            out[dst] = (("time", "station"), ds0[src].values)

    return out


# ============================================================
# READ HURRYWAVE BULK FROM _his FILE
# ============================================================

def find_hurrywave_station_label_var(ds):
    candidates = [
        "station_name",
        "stations_name",
        "station_names",
        "station",
        "stations",
        "site_name",
        "name",
        "point_name",
    ]

    for v in candidates:
        if v in ds.variables and ds[v].ndim in [1, 2]:
            return v
    for v in candidates:
        if v in ds.coords and ds.coords[v].ndim in [1, 2]:
            return v
    return None


def read_hurrywave_bulk_buoys(his_nc_path, nstations):
    """
    Read first nstations from HurryWave his file.

    Expected variable names:
      point_hm0
      point_tp
      point_wavdir
    """
    ds0 = xr.open_dataset(his_nc_path, mask_and_scale=True, decode_cf=True)
    time_name = get_time_name(ds0)

    hm0_name = "point_hm0"
    tp_name = "point_tp"
    dir_name = "point_wavdir"

    for name in [hm0_name, tp_name, dir_name]:
        if name not in ds0:
            raise KeyError(f"{name} not found in {his_nc_path}")

    dims = ds0[hm0_name].dims
    station_dim = None
    for d in dims:
        if d != time_name:
            station_dim = d
            break
    if station_dim is None:
        raise ValueError(f"Could not infer station dimension from {hm0_name}")

    hm0 = ds0[hm0_name].isel({station_dim: slice(0, nstations)})
    tp = ds0[tp_name].isel({station_dim: slice(0, nstations)})
    wdir = ds0[dir_name].isel({station_dim: slice(0, nstations)})

    out = xr.Dataset(
        coords={
            "time": ds0[time_name].values,
            "station": np.arange(hm0.sizes[station_dim]),
        },
        data_vars={
            "hm0_hw": (("time", "station"), hm0.values),
            "tp_hw": (("time", "station"), tp.values),
            "wdir_from_hw": (("time", "station"), wdir.values),
        },
    )

    label_var = find_hurrywave_station_label_var(ds0)
    if label_var is not None:
        try:
            labels = decode_string_array(ds0[label_var].isel({station_dim: slice(0, nstations)}).values)
            out["station_label_hw"] = ("station", labels)
            out["station_id_guess"] = (
                "station",
                np.array([extract_ndbc_id(lbl) or "" for lbl in labels], dtype=object),
            )
        except Exception:
            pass

    return out


# ============================================================
# MATCH HURRYWAVE TO NDBC
# ============================================================

def build_station_match_table(ds_hw, ds_buoy):
    """
    Prefer matching by station_id extracted from HurryWave labels.
    If that fails, fall back to matching the first N in order.
    """
    buoy_ids = decode_string_array(ds_buoy["station_id"].values)
    buoy_names = decode_string_array(ds_buoy["station_name"].values)

    rows = []

    if "station_id_guess" in ds_hw:
        hw_ids = decode_string_array(ds_hw["station_id_guess"].values)
        hw_labels = decode_string_array(ds_hw["station_label_hw"].values) if "station_label_hw" in ds_hw else np.array([""] * ds_hw.sizes["station"], dtype=object)

        buoy_lookup = {sid: j for j, sid in enumerate(buoy_ids)}

        for i, sid in enumerate(hw_ids):
            if sid and sid in buoy_lookup:
                j = buoy_lookup[sid]
                rows.append({
                    "hw_station": i,
                    "buoy_station": j,
                    "station_id": sid,
                    "station_label_hw": hw_labels[i],
                    "station_name_buoy": buoy_names[j],
                    "longitude": float(ds_buoy["longitude"].isel(station=j).item()) if "longitude" in ds_buoy else np.nan,
                    "latitude": float(ds_buoy["latitude"].isel(station=j).item()) if "latitude" in ds_buoy else np.nan,
                    "water_depth_m": float(ds_buoy["water_depth_m"].isel(station=j).item()) if "water_depth_m" in ds_buoy else np.nan,
                    "match_method": "station_id_from_hurrywave_label",
                })

    match_df = pd.DataFrame(rows)

    if len(match_df) == 0:
        n = min(ds_hw.sizes["station"], ds_buoy.sizes["station"])
        hw_labels = decode_string_array(ds_hw["station_label_hw"].values) if "station_label_hw" in ds_hw else np.array([""] * n, dtype=object)

        rows = []
        for i in range(n):
            rows.append({
                "hw_station": i,
                "buoy_station": i,
                "station_id": buoy_ids[i],
                "station_label_hw": hw_labels[i] if i < len(hw_labels) else "",
                "station_name_buoy": buoy_names[i],
                "longitude": float(ds_buoy["longitude"].isel(station=i).item()) if "longitude" in ds_buoy else np.nan,
                "latitude": float(ds_buoy["latitude"].isel(station=i).item()) if "latitude" in ds_buoy else np.nan,
                "water_depth_m": float(ds_buoy["water_depth_m"].isel(station=i).item()) if "water_depth_m" in ds_buoy else np.nan,
                "match_method": "fallback_order",
            })

        match_df = pd.DataFrame(rows)

    return match_df


# ============================================================
# BUILD MATCHED TIME SERIES
# ============================================================

def build_pair_dataframe(ds_hw, ds_buoy, hw_station, buoy_station, time_tol="30min"):
    df_hw = pd.DataFrame({
        "time": pd.to_datetime(ds_hw["time"].values),
        "hm0_hw": ds_hw["hm0_hw"].isel(station=hw_station).values,
        "tp_hw": ds_hw["tp_hw"].isel(station=hw_station).values,
        "wdir_from_hw": ds_hw["wdir_from_hw"].isel(station=hw_station).values,
    }).sort_values("time")

    df_buoy = pd.DataFrame({
        "time": pd.to_datetime(ds_buoy["time"].values),
        "wvht_buoy": ds_buoy["wvht_buoy"].isel(station=buoy_station).values,
        "dpd_buoy": ds_buoy["dpd_buoy"].isel(station=buoy_station).values,
        "apd_buoy": ds_buoy["apd_buoy"].isel(station=buoy_station).values,
        "mwd_from_buoy": ds_buoy["mwd_from_buoy"].isel(station=buoy_station).values,
    }).sort_values("time")

    df = pd.merge_asof(
        df_hw,
        df_buoy,
        on="time",
        direction="nearest",
        tolerance=pd.Timedelta(time_tol),
    )

    return df


# ============================================================
# SUMMARY STATS
# ============================================================

def paired_stats(df, use_period="dpd"):
    if use_period.lower() == "apd":
        period_buoy = df["apd_buoy"].values
        period_name = "APD"
    else:
        period_buoy = df["dpd_buoy"].values
        period_name = "DPD"

    d_hm0 = df["hm0_hw"].values - df["wvht_buoy"].values
    d_tp = df["tp_hw"].values - period_buoy
    d_dir = circ_diff_deg(df["wdir_from_hw"].values, df["mwd_from_buoy"].values)

    return pd.Series({
        "n_hm0": int(np.sum(np.isfinite(df["hm0_hw"]) & np.isfinite(df["wvht_buoy"]))),
        f"{period_name}_n": int(np.sum(np.isfinite(df["tp_hw"]) & np.isfinite(period_buoy))),
        "dir_n": int(np.sum(np.isfinite(df["wdir_from_hw"]) & np.isfinite(df["mwd_from_buoy"]))),
        "Hm0 bias (m)": float(np.nanmean(d_hm0)),
        "Hm0 RMSE (m)": float(rmse(d_hm0)),
        f"Tp-{period_name} bias (s)": float(np.nanmean(d_tp)),
        f"Tp-{period_name} RMSE (s)": float(rmse(d_tp)),
        "Dir bias (deg)": float(np.nanmean(d_dir)),
        "Dir RMSE (deg)": float(rmse(d_dir)),
    })


def build_all_pair_stats(ds_hw, ds_buoy, match_df, use_period="dpd", time_tol="30min"):
    rows = []
    for _, r in match_df.iterrows():
        df = build_pair_dataframe(
            ds_hw,
            ds_buoy,
            hw_station=int(r["hw_station"]),
            buoy_station=int(r["buoy_station"]),
            time_tol=time_tol,
        )
        st = paired_stats(df, use_period=use_period).to_dict()
        st.update({
            "hw_station": int(r["hw_station"]),
            "buoy_station": int(r["buoy_station"]),
            "station_id": r["station_id"],
            "station_name_buoy": r["station_name_buoy"],
            "water_depth_m": r["water_depth_m"],
            "match_method": r["match_method"],
        })
        rows.append(st)
    return pd.DataFrame(rows)


# ============================================================
# PLOTTING
# ============================================================

def plot_one_station_comparison(df, station_label="", use_period="dpd"):
    if use_period.lower() == "apd":
        buoy_period_col = "apd_buoy"
        buoy_period_label = "Buoy APD"
    else:
        buoy_period_col = "dpd_buoy"
        buoy_period_label = "Buoy DPD"

    fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

    ax[0].plot(df["time"], df["hm0_hw"], ".-", label="HurryWave Hm0")
    ax[0].plot(df["time"], df["wvht_buoy"], ".-", label="NDBC WVHT")
    ax[0].set_ylabel("Hm0 (m)")
    ax[0].grid(True)
    ax[0].legend()

    ax[1].plot(df["time"], df["tp_hw"], ".-", label="HurryWave Tp")
    ax[1].plot(df["time"], df[buoy_period_col], ".-", label=buoy_period_label)
    ax[1].set_ylabel("Period (s)")
    ax[1].grid(True)
    ax[1].legend()

    ax[2].plot(df["time"], df["wdir_from_hw"], ".-", label="HurryWave dir (from)")
    ax[2].plot(df["time"], df["mwd_from_buoy"], ".-", label="NDBC MWD")
    ax[2].set_ylabel("Direction (deg)")
    ax[2].set_xlabel("Time")
    ax[2].set_ylim(0, 360)
    ax[2].set_yticks([0, 90, 180, 270, 360])
    ax[2].grid(True)
    ax[2].legend()

    fig.suptitle(station_label)
    fig.tight_layout()
    return fig, ax


def save_all_comparison_plots(ds_hw, ds_buoy, match_df, outdir, use_period="dpd", time_tol="30min"):
    ensure_dir(outdir)

    for _, r in match_df.iterrows():
        df = build_pair_dataframe(
            ds_hw,
            ds_buoy,
            hw_station=int(r["hw_station"]),
            buoy_station=int(r["buoy_station"]),
            time_tol=time_tol,
        )

        label = f"{r['station_id']} | depth={r['water_depth_m']:.1f} m | {r['station_name_buoy']}"
        fig, ax = plot_one_station_comparison(df, station_label=label, use_period=use_period)

        out_png = os.path.join(outdir, f"compare_hurrywave_ndbc_{r['station_id']}.png")
        fig.savefig(out_png, dpi=150, bbox_inches="tight")
        plt.close(fig)

    print(f"Saved comparison plots to: {outdir}")


# ============================================================
# RUN
# ============================================================
NDBC_WAVE_DATA_DIR = "F:/crs/proj/2025_NOPP_comparison/helene_waves/"
HURRYWAVE_DATA_DIR = "F:/crs/proj/2025_NOPP_comparison/helene_deltares_wave_model_output/helene89pervmax/"
PLOTDIR = "F:/crs/proj/2025_NOPP_comparison/helene_waves/"
OUTDIR_CSV = "F:/crs/proj/2025_NOPP_comparison/helene_waves/"

META_CSV = "ndbc_buoy_list.csv"
HURRYWAVE_HIS_NC   = "hurrywave_his.nc"

meta = read_buoy_metadata(NDBC_WAVE_DATA_DIR+META_CSV)
print("Metadata table:")
print(meta)
print()

df_ndbc, per_station = download_ndbc_bulk_for_list(meta, start=START, end=END, year=YEAR)
print()

save_station_csvs(per_station, OUTDIR_CSV)

ds_ndbc = save_ndbc_to_netcdf(df_ndbc, meta, NDBC_WAVE_DATA_DIR+OUT_NC)
print(ds_ndbc)
print()

# comparison with HurryWave bulk stats
ds_buoy = read_ndbc_bulk_nc(NDBC_WAVE_DATA_DIR+OUT_NC)
ds_hw = read_hurrywave_bulk_buoys(HURRYWAVE_DATA_DIR+HIS_NC, nstations=len(meta))

match_df = build_station_match_table(ds_hw, ds_buoy)
print("Matched stations:")
print(match_df)
print()

stats_df = build_all_pair_stats(
    ds_hw,
    ds_buoy,
    match_df,
    use_period=USE_PERIOD,
    time_tol=TIME_TOL,
)
print("Comparison statistics:")
print(stats_df)
print()

if MAKE_PLOTS and len(match_df) > 0:
    save_all_comparison_plots(
        ds_hw,
        ds_buoy,
        match_df,
        outdir=PLOTDIR,
        use_period=USE_PERIOD,
        time_tol=TIME_TOL,
    )
else:
    print("No plots made.")

Metadata table:
  station_id                                       station_name  longitude  \
0      42002  NDBC 42002 (LLNR 1470) - West Gulf - 207 nm Ea...    -93.780   
1      42012  NDBC 42012 (LLNR 138) Orange Beach - 44 nm SE ...    -87.548   
2      42013                NDBC 42013 - C10 - WFS Central Buoy    -82.924   
3      42022                NDBC 42022 - C12 - WFS Central Buoy    -83.741   
4      42036  NDBC 42036 (LLNR 855)- West Tampa - 112 nm WNW...    -87.508   
5      42040  NDBC 42040 (LLNR 245) - Luke Offshore Test Pla...    -88.237   
6      42097                          NDBC 42097 - Pulley Ridge    -83.654   
7      42098               NDBC 42098 - Egmont Channel Entrance    -82.931   

   latitude  water_depth_m  
0    25.950         3208.0  
1    30.060           23.5  
2    27.173           25.0  
3    27.505           50.0  
4    28.501           50.9  
5    29.207          192.0  
6    25.704           81.0  
7    27.590           14.0  

42002: 720 records


C:\Users\csherwood\AppData\Local\miniforge3\envs\CRS\lib\site-packages\xarray\backends\api.py:667: RuntimeWarning: 'netcdf4' fails while guessing
  engine = plugins.guess_engine(filename_or_obj)
C:\Users\csherwood\AppData\Local\miniforge3\envs\CRS\lib\site-packages\xarray\backends\api.py:667: RuntimeWarning: 'h5netcdf' fails while guessing
  engine = plugins.guess_engine(filename_or_obj)
C:\Users\csherwood\AppData\Local\miniforge3\envs\CRS\lib\site-packages\xarray\backends\api.py:667: RuntimeWarning: 'scipy' fails while guessing
  engine = plugins.guess_engine(filename_or_obj)


ValueError: did not find a match in any of xarray's currently installed IO backends ['netcdf4', 'h5netcdf', 'scipy', 'rasterio']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html
https://docs.xarray.dev/en/stable/user-guide/io.html

In [7]:
print(HURRYWAVE_DATA_DIR+HIS_NC)

F:/crs/proj/2025_NOPP_comparison/helene_deltares_wave_model_output/helene89pervmax/F:/crs/proj/2025_NOPP_comparison/helene_deltares_wave_model_output/helene89pervmax/hurrywave_his.nc
